# Data Collection

This notebook fetches macroeconomic and financial data from **Yahoo Finance** and **FRED** (Federal Reserve Economic Data) starting from **January 1990** to **December 2025**. The extended date range covers key economic events such as the dot-com bubble, the 2008 financial crisis, and the COVID-19 pandemic.

| Variable | Source | Series Code |
|---|---|---|
| S&P 500 Index | Yahoo Finance | ^GSPC |
| S&P 500 Energy | Yahoo Finance | XLE |
| S&P 500 IT | Yahoo Finance | XLK |
| S&P 500 Financials | Yahoo Finance | XLF |
| S&P 500 Healthcare | Yahoo Finance | XLV |
| S&P 500 Industrials | Yahoo Finance | XLI |
| CPI | FRED | CPIAUCSL |
| Fed Funds Rate | FRED | FEDFUNDS |
| Unemployment Rate | FRED | UNRATE |
| GDP Growth | FRED | A191RL1Q225SBEA |
| Exchange Rate (USD Index) | Yahoo Finance | DX-Y.NYB |



## 1. Install & Import Libraries

In [19]:
# Install required packages (run once)
!pip install yfinance pandas pandas-datareader fredapi python-dotenv -q


In [20]:
import yfinance as yf                  # For fetching S&P 500 data from Yahoo Finance
import pandas_datareader.data as web    # For fetching data from FRED
from datetime import datetime           # For handling dates
import pandas as pd                     # For data manipulation
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path().resolve().parent
sys.path.append(str(project_root))

## 2. Define Date Range

We collect data from **January 2000** up to **December 2025** to ensure we have enough historical data for analysis. You can adjust `START` and `END` as needed.

In [21]:
# Set the date range for all data fetches
START = "1990-01-01"
END = "2025-12-31"

print(f"Date range: {START} to {END}")

Date range: 1990-01-01 to 2025-12-31


## 3. Fetch S&P 500 Sector Indices (Yahoo Finance)

We fetch the **Close** prices for 6 key S&P 500 sector indices from Yahoo Finance:

| Sector | Ticker | Description |
|---|---|---|
| S&P 500 | `^GSPE` | S&P 500 Sector Index |
| Energy | `XLE` | S&P 500 Energy Sector Index |
| Information Technology | `XLK` | S&P 500 IT Sector Index |
| Financials | `XLF` | S&P 500 Financials Sector Index |
| Healthcare | `XLV` | S&P 500 Healthcare Sector Index |
| Industrials | `XLI` | S&P 500 Industrials Sector Index |


> 

In [22]:
# Define S&P 500 sector indices tickers and column names
indices = {
    "SP500": "^GSPC",            
    "SP500_Energy": "XLE",     
    "SP500_IT": "XLK",    
    "SP500_Financials": "XLF",  
    "SP500_Healthcare": "XLV",   
    "SP500_Industrials": "XLI"
}

# Download all sector indices from Yahoo Finance

all_data = []

for name, ticker in indices.items():
    
    print(f"Downloading data for {name} ({ticker})...")
    
    # Download data with a monthly interval
    data = yf.download(
        ticker,
        start=START,
        end=END,
        interval="1mo",   # Monthly interval
        auto_adjust=False,
        progress=False
    )
    
    # Drop rows with all NaN values (if any)
    # data.dropna(how="all", inplace=True)
    
    # Save to CSV file
    file_name = f"{name}_monthly_data.csv"

    # Remove multi-index if exists
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
        
    data.to_csv(f"../data/SP500/{file_name}",index=True)
    
    print(f"Saved: {file_name}")

print("\nAll indices downloaded successfully!")

Saved: SP500_monthly_data.csv
Saved: SP500_Energy_monthly_data.csv
Saved: SP500_IT_monthly_data.csv
Saved: SP500_Financials_monthly_data.csv
Saved: SP500_Healthcare_monthly_data.csv
Saved: SP500_Industrials_monthly_data.csv

All indices downloaded successfully!


In [23]:
from src.data_collection import merge_monthly_csv_files

# Merge data
merged_df = merge_monthly_csv_files("../data/SP500")

merged_df.head()

,Date,SP500_Energy_Open,SP500_Energy_Close,SP500_Energy_Volume,SP500_Healthcare_Open,SP500_Healthcare_Close,SP500_Healthcare_Volume,SP500_Industrials_Open,SP500_Industrials_Close,SP500_Industrials_Volume,SP500_Open,SP500_Close,SP500_Volume,SP500_Financials_Open,SP500_Financials_Close,SP500_Financials_Volume,SP500_IT_Open,SP500_IT_Close,SP500_IT_Volume
0,1990-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,353.399994,329.079987,3793250000,NaN,NaN,NaN,NaN,NaN,NaN
1,1990-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,329.079987,331.890015,2961970000,NaN,NaN,NaN,NaN,NaN,NaN
2,1990-03-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,331.890015,339.940002,3283280000,NaN,NaN,NaN,NaN,NaN,NaN
3,1990-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,339.940002,330.799988,2801220000,NaN,NaN,NaN,NaN,NaN,NaN
4,1990-05-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,330.799988,361.230011,3596680000,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Save processed dataset
merged_df.to_csv("../data/SP500_monthly.csv", index=False)

## 4. Fetch FRED Data

We fetch 5 macroeconomic variables from FRED using `pandas_datareader` (from 2000 to 2025):

- CPIAUCSL — Consumer Price Index for All Urban Consumers (monthly)
- FEDFUNDS — Effective Federal Funds Rate (monthly)
- UNRATE — Unemployment Rate (monthly)
- A191RL1Q225SBEA — Real GDP Growth Rate (quarterly, annualised)
- VIXCLS — VIX measures market expectation of near term volatility
- AAA - Moody's Seasoned Aaa Corporate Bond Yield(Highest credit quality)
- BBB - Moody's Seasoned Baa Corporate Bond Yield

In [25]:
# Define FRED series codes and their readable names (excluding USD Index — handled separately)
fred_series = {
    "CPI": "CPIAUCSL",
    "Interest_Rate": "FEDFUNDS",
    "GDP": "GDP",
    "Unemployment": "UNRATE",
    "VIX": "VIXCLS",
    "BAA": "BAA",
    "AAA": "AAA"
}

In [27]:
# Fetch each FRED series and store in a list
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env and loads vars into environment

api_key = os.getenv("API_KEY")

from fredapi import Fred

data = pd.DataFrame()

fred = Fred(api_key=api_key)

for name, series in fred_series.items():
    temp = fred.get_series(
        series,
        observation_start=START,
        observation_end=END
    )
    
    temp.index = pd.to_datetime(temp.index)


    if(name == "VIXCLS"):
        temp = temp.resample("MS").mean()  # Resample to monthly frequency (start of month)

    data[name] = temp

data["Credit_Spread"] = data["BAA"] - data["AAA"]

data.to_csv("../data/macro_data_fred.csv")


print("Data saved successfully.")

Data saved successfully.


In [ ]:
data.head(10)

,CPI,Interest_Rate,GDP,Unemployment,VIX,BAA,AAA,Credit_Spread
1990-01-01,127.5,8.23,5872.701,5.4,23.347273,9.94,8.99,0.95
1990-02-01,128.0,8.24,NaN,5.3,23.262632,10.14,9.22,0.92
1990-03-01,128.6,8.28,NaN,5.2,20.062273,10.21,9.37,0.84
1990-04-01,128.9,8.26,5960.028,5.4,21.403500,10.30,9.46,0.84
1990-05-01,129.1,8.18,NaN,5.4,18.097727,10.41,9.47,0.94
1990-06-01,129.9,8.29,NaN,5.2,16.822381,10.22,9.26,0.96
1990-07-01,130.5,8.15,6015.116,5.5,18.392857,10.20,9.24,0.96
1990-08-01,131.6,8.13,NaN,5.7,28.175217,10.41,9.41,1.00
1990-09-01,132.5,8.20,NaN,5.9,29.107368,10.64,9.56,1.08
1990-10-01,133.4,8.11,6004.733,5.9,29.625652,10.74,9.53,1.21


#### Fetch USD Index

The Broad USD Index (`DTWEXBGS`) only starts from **2006**. To get coverage from **2000**, we use USD Index (`DX-Y.NYB`, available 1973).

In [ ]:

usd = yf.download("DX-Y.NYB", start=START, end=END, auto_adjust=False)

usd.index = pd.to_datetime(usd.index)

# Force Close column to Series safely
usd_close = usd.loc[:, "Close"]

# If it is still DataFrame, squeeze it
usd_close = usd_close.squeeze()

# Now resample
usd_monthly = usd_close.resample("MS").mean()

# Convert to DataFrame only if needed
if isinstance(usd_monthly, pd.Series):
    usd_monthly = usd_monthly.to_frame(name="USD_Index")
else:
    usd_monthly.columns = ["USD_Index"]

usd_monthly.to_csv("../data/usd_index_monthly.csv")

print("USD Index downloaded successfully.")

[*********************100%***********************]  1 of 1 completed

USD Index downloaded successfully.


## 5. Merge All Data

The datasets have different frequencies (daily, monthly, quarterly). We resample everything to **monthly frequency** (end-of-month) so they can be merged into a single DataFrame spanning from **2000 to 2025**.

For the merge, we use macro_data_fred.csv, usd_index_monthly.csv,SP500_Master_Monthly.csv files

In [ ]:

# Load CSV files
macro_data = pd.read_csv("../data/macro_data_fred.csv", parse_dates=True, index_col=0)
usd_data = pd.read_csv("../data/usd_index_monthly.csv", parse_dates=True, index_col='Date')
sp500_data = pd.read_csv("../data/SP500_monthly.csv", parse_dates=True, index_col='Date')

df = macro_data.join(sp500_data, how='outer').join(usd_data, how='outer')
df.index.name = 'Date'
df.sort_index(inplace=True)

df.to_csv("../data/processed/main_data.csv")

## 6. Quick Data Overview

In [ ]:
# Display basic statistics
df.describe()



,CPI,Interest_Rate,GDP,Unemployment,VIX,BAA,AAA,Credit_Spread,SP500_Energy_Open,SP500_Energy_Close,...,SP500_Open,SP500_Close,SP500_Volume,SP500_Financials_Open,SP500_Financials_Close,SP500_Financials_Volume,SP500_IT_Open,SP500_IT_Close,SP500_IT_Volume,USD_Index
count,431.000000,432.000000,144.000000,431.000000,432.000000,432.000000,432.000000,432.000000,325.000000,325.000000,...,432.000000,432.000000,4.320000e+02,325.000000,325.000000,3.250000e+02,325.000000,325.000000,3.250000e+02,432.000000
mean,209.994677,2.879259,15031.346111,5.677262,19.460239,6.542755,5.604167,0.938588,29.650085,29.572133,...,1812.116429,1826.440090,5.399759e+10,23.945779,24.037986,1.153343e+09,32.254265,32.277571,2.817167e+08,92.505021
std,51.524574,2.343193,6779.638558,1.741716,7.416527,1.736061,1.750633,0.365919,11.437541,10.926330,...,1434.670608,1453.001531,3.787546e+10,9.639184,9.792590,1.239635e+09,32.777557,32.040947,2.201511e+08,10.066577
min,127.500000,0.050000,5872.701000,3.400000,10.125455,3.160000,2.140000,0.550000,10.600000,10.580000,...,303.989990,304.000000,2.687280e+09,5.938262,6.173842,5.512420e+05,5.970000,5.915000,2.976800e+06,72.116819
25%,164.550000,0.235000,9382.259250,4.400000,14.083095,5.190000,4.070000,0.700000,18.889999,18.950001,...,919.845001,919.275009,1.569714e+10,18.188465,18.204712,1.628039e+08,11.255000,11.200000,6.248060e+07,84.481610
50%,211.398000,2.620000,14439.892000,5.400000,17.673258,6.320000,5.400000,0.870000,30.990000,31.350000,...,1302.359985,1302.829956,6.421550e+10,22.867586,22.851339,9.450828e+08,18.209999,18.225000,2.798902e+08,92.645660
75%,242.331500,5.220000,19137.055250,6.600000,22.979259,7.950000,7.085000,1.050000,38.139999,38.220001,...,2213.019958,2248.840088,8.215330e+10,28.082859,27.950001,1.390802e+09,37.360001,37.665001,4.125822e+08,98.837040
max,326.031000,8.290000,31490.070000,14.800000,62.668947,10.740000,9.560000,3.380000,90.940002,50.049999,...,6882.319824,6849.089844,1.621854e+11,53.700001,54.770000,7.904224e+09,217.440002,150.339996,1.564823e+09,118.966501


In [ ]:
df.columns

Index(['CPI', 'Interest_Rate', 'GDP', 'Unemployment', 'VIX', 'BAA', 'AAA',
       'Credit_Spread', 'SP500_Energy_Open', 'SP500_Energy_Close',
       'SP500_Energy_Volume', 'SP500_Healthcare_Open',
       'SP500_Healthcare_Close', 'SP500_Healthcare_Volume',
       'SP500_Industrials_Open', 'SP500_Industrials_Close',
       'SP500_Industrials_Volume', 'SP500_Open', 'SP500_Close', 'SP500_Volume',
       'SP500_Financials_Open', 'SP500_Financials_Close',
       'SP500_Financials_Volume', 'SP500_IT_Open', 'SP500_IT_Close',
       'SP500_IT_Volume', 'USD_Index'],
      dtype='object')

In [ ]:
# Check for missing values in each column
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal rows: {len(df)}")


Missing values per column:
CPI                           1
Interest_Rate                 0
GDP                         288
Unemployment                  1
VIX                           0
BAA                           0
AAA                           0
Credit_Spread                 0
SP500_Energy_Open           107
SP500_Energy_Close          107
SP500_Energy_Volume         107
SP500_Healthcare_Open       107
SP500_Healthcare_Close      107
SP500_Healthcare_Volume     107
SP500_Industrials_Open      107
SP500_Industrials_Close     107
SP500_Industrials_Volume    107
SP500_Open                    0
SP500_Close                   0
SP500_Volume                  0
SP500_Financials_Open       107
SP500_Financials_Close      107
SP500_Financials_Volume     107
SP500_IT_Open               107
SP500_IT_Close              107
SP500_IT_Volume             107
USD_Index                     0
dtype: int64

Total rows: 432
